# Experimentación de modelos para Marca 2
Este notebook ejecuta la lógica OOP del paquete y conserva las tablas y gráficos para la entrega.


In [ ]:
from pathlib import Path
import pandas as pd
from volume_forecasting.config import ConfigLoader
from volume_forecasting.data_loader import DataLoader
from volume_forecasting.feature_engineering import FeatureEngineer
from volume_forecasting.experiment import ForecastExperiment
from volume_forecasting.modeling import ProductionModelTrainer
from volume_forecasting.artifacts import ArtifactStore


In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
config = ConfigLoader.load(PROJECT_ROOT / "config/model_config.json")
sales, macro = DataLoader().load_sources(PROJECT_ROOT / config["sales_path"], PROJECT_ROOT / config["macro_path"], config["sales_sheet"])
data = FeatureEngineer(brand=config["brand"]).build(sales, macro)
assert data["Marca"].eq("Marca 2").all()
display(data.head())


In [ ]:
feature_sets = ForecastExperiment.feature_sets_default()
experiment = ForecastExperiment(feature_sets, test_months=12, max_abs_bias=0.07)
output = experiment.run(data)
display(output.results)
display(output.selected_result)
display(output.accuracy_result)


## Exportación
El modelo operativo se reentrena con toda la historia disponible antes de guardarse.


In [ ]:
selected = output.selected_result
features = feature_sets[selected["Feature_Set"]]
production_model, production_rows = ProductionModelTrainer.refit(output.selected_model, data, features)
metadata = {"brand":"Marca 2", "model_key":f"{selected['Feature_Set']}__{selected['Model']}", "features":features, "target":"Log_Volume", "holdout_metrics":{"wape":selected["WAPE"], "bias":selected["BIAS"], "mae":selected["MAE"], "rmse":selected["RMSE"]}, "refit_on_all_available_data":True, "production_training_rows":production_rows}
ArtifactStore.save(production_model, metadata, PROJECT_ROOT / "models/best_model.joblib", PROJECT_ROOT / "models/model_metadata.json")
